# Graph Builder


This is the fourth preliminary step for training. This is where we build our graph and specify graph properties such as

- whether the graph is heterogenous / homogenous
- node types
- edge types
- whether an edge should exist between nodes
- node class
- etc.

```{eval-rst}
.. autoclass:: src.data.GraphBuilder
    :members:
    :undoc-members:
    :show-inheritance:
```




## GraphDataset

This section how Deepsnap handles a graph dataset. Particularly in how the edges and node are represented.

In [49]:
import networkx as nx
import polars as pl
from deepsnap.dataset import GraphDataset
from src.data import DatasetLoader, GraphBuilder, Preprocessor
from src.models import Features

dataset_loader = DatasetLoader()
data: pl.DataFrame = dataset_loader.load_dataset()
preprocessor = Preprocessor(data=data)
data = preprocessor()

node_features = Features(
    tweet=[
        "favorite_count",
        # "retweet_count",
        # "bookmark_count",
        "reply_count",
        "quote_count",
        # "views",
        "source",
        "is_hateful",
    ],
    user=[
        "favourites_count",
        "follower_count",
        "following_count",
        # "number_of_tweets",
        # "listed_count",
        # "is_blue_verified",
        # "friends",
    ],
)

gb = GraphBuilder(data=data, node_features=node_features)

graph: nx.DiGraph | nx.Graph = gb.create_graph(directed=True)

dataset = GraphDataset([graph], task="link_pred", edge_train_mode="disjoint")

✅ Loading cached dataset - Done.
✅ Loading dataset - Done.
✅ Categorizing source column - Done.


In [50]:
dataset

GraphDataset(1)

In [51]:
ds_graph = dataset[0]
ds_graph

Graph(G=[], edge_index=[2, 36365], edge_label_index=[2, 36365], negative_label_val=[1], node_feature=[61682, 5], node_label=[61682], node_label_index=[61682], node_type=[61682], task=[])

In [52]:
type(ds_graph)

deepsnap.graph.Graph

As you can see we are supposed to create a deepsnap dataset. `dataset` is an iterable of deepsnap graphs.


### Nodes

DeepSnap stores features as a tensor of features. We have 61282 samples with 5 features.

In [53]:
ds_graph.node_feature.shape

torch.Size([61682, 5])

In [54]:
ds_graph.node_feature

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [9.4100e+02, 3.9000e+01, 6.0000e+00, 0.0000e+00, 0.0000e+00],
        [4.8600e+03, 3.3400e+02, 2.8000e+01, 0.0000e+00, 1.0000e+00],
        ...,
        [4.8000e+01, 1.9500e+02, 4.3000e+01, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [1.1259e+04, 6.0700e+02, 2.6590e+03, 0.0000e+00, 0.0000e+00]])

### Edges

Edges are represented in COO (Coordinate) format

In [55]:
ds_graph.edge_index

tensor([[    1,     3,     3,  ..., 61677, 61679, 61681],
        [    0,     2, 13238,  ..., 61676, 61678, 61680]])

In [56]:
ds_graph.edge_label_index

tensor([[    1,     3,     3,  ..., 61677, 61679, 61681],
        [    0,     2, 13238,  ..., 61676, 61678, 61680]])

In [57]:
ds_graph.edge_label_index[0, :] # Another way of indexing the source nodes

tensor([    1,     3,     3,  ..., 61677, 61679, 61681])

In COO format, the first tensor contains the source nodes. The second tensor contains the target nodes. Each number is the corresponding index of a feature in `ds_graph.node_feature`, however when the graph is undirected, the order doesn't matter. `edge_index` and `edge_label_index` are equal in this case because while building the graph, we haven't specified anything that warrant separate label representation. You can use either of them.

So when we have:

In [ ]:
# edge_index / edge_label_index
[
    [1, ...], # Source nodes
    [0, ...] # Target nodes
]

This node source's target is

In [ ]:
ds_graph.node_feature[1] # Feature of node 1, a "user0" node

tensor([941.,  39.,   6.,   0.,   0.])

This one

In [60]:
ds_graph.node_feature[0] # Features of node 0, a "tweet" node

tensor([0., 0., 0., 0., 1.])

### Example usage

Let's try doing a dot product of node source node and target node.

In [61]:
import torch

torch.index_select(ds_graph.node_feature, dim=0, index=ds_graph.edge_index[0])

tensor([[9.4100e+02, 3.9000e+01, 6.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.3269e+04, 9.5956e+04, 1.5960e+03, 0.0000e+00, 0.0000e+00],
        [1.3269e+04, 9.5956e+04, 1.5960e+03, 0.0000e+00, 0.0000e+00],
        ...,
        [3.0710e+03, 5.4046e+04, 5.2576e+04, 0.0000e+00, 0.0000e+00],
        [4.8000e+01, 1.9500e+02, 4.3000e+01, 0.0000e+00, 0.0000e+00],
        [1.1259e+04, 6.0700e+02, 2.6590e+03, 0.0000e+00, 0.0000e+00]])

We can easily select specific feature nodes in an iterable by passing a list of indexes to `torch.index_select`

Here we display an example on how we can access feature information using edge indexes

In [62]:
sources = ds_graph.edge_index[0]
example_node_source = sources[0]

torch.equal(torch.index_select(ds_graph.node_feature, dim=0, index=sources)[0], (ds_graph.node_feature[example_node_source]))

True

In [63]:
torch.index_select(ds_graph.node_feature, dim=0, index=sources)[0]

tensor([941.,  39.,   6.,   0.,   0.])

In [64]:
ds_graph.node_feature[example_node_source]

tensor([941.,  39.,   6.,   0.,   0.])

Now lets do the dot product.

In [65]:
source_features = torch.index_select(ds_graph.node_feature, dim=0, index=ds_graph.edge_index[0])
target_features = torch.index_select(ds_graph.node_feature, dim=0, index=ds_graph.edge_index[1])

dot_products = torch.sum(source_features * target_features, dim=1)

dot_products

tensor([0.0000e+00, 9.6581e+07, 2.6504e+06,  ..., 0.0000e+00, 4.3200e+02,
        1.1259e+04])

The shape we get is exactly the total amount of edges.

In [66]:
dot_products.shape

torch.Size([36365])